# Proyecto 5. Clasificación de flores

**Objetivo:** Adaptar ResNet18 preentrenada mediante transferencia de aprendizaje.

Ejecuten las celdas en orden. Después de **cada celda de código**, respondan las preguntas en la celda de texto siguiente. Guarden evidencia de sus experimentos y justifiquen cada decisión con resultados y fuentes consultadas. Reinicien el entorno y ejecuten todo antes de entregar.

**Entorno:** Google Colab o Python con Jupyter. Instalen una versión compatible de PyTorch y torchvision, además de numpy, matplotlib, pandas, scikit-learn y datasets cuando corresponda. En Colab pueden usar una celda separada con `%pip install torch torchvision numpy matplotlib pandas scikit-learn datasets`; reinicien el entorno si se les solicita. Descargar datos o pesos requiere internet. Oxford Flowers 102 y pesos de ResNet18 se descargan desde torchvision; requiere internet y espacio en disco.

## 1. Importaciones y reproducibilidad

Importamos las herramientas y fijamos una semilla. Los resultados pueden variar según hardware y versiones.

In [ ]:
import random  # Controla el generador aleatorio estándar.
import numpy as np  # Proporciona arreglos y operaciones numéricas.
import matplotlib.pyplot as plt  # Permite construir gráficas.
import torch  # Proporciona tensores y operaciones para redes neuronales.
from torch import nn  # Contiene capas y funciones de pérdida.
from torch.utils.data import DataLoader, Dataset, random_split  # Permite crear lotes y conjuntos de datos.
from pathlib import Path  # Facilita el manejo de rutas.
random.seed(42)  # Fija la semilla de Python.
np.random.seed(42)  # Fija la semilla de NumPy.
torch.manual_seed(42)  # Fija la semilla inicial de PyTorch.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')  # Selecciona GPU cuando está disponible.
print('Dispositivo:', device, '| PyTorch:', torch.__version__)  # Muestra el entorno de ejecución.


### Preguntas y actividad obligatoria

1. ¿Qué aporta cada biblioteca? Investiguen la documentación de `torch.device`.
2. Cambien la semilla, ejecuten el proyecto y comparen resultados; ¿garantiza repetir exactamente todos los cálculos?

**Respuestas del equipo:** escriban aquí sus respuestas y evidencias.

## 2. Datos y conjuntos

Preparen entrenamiento, validación y prueba sin mezclar información entre ellos. La validación sirve para decisiones; la prueba se reserva para la evaluación final.

In [ ]:
from torchvision import datasets, transforms, models  # Importa datasets, transformaciones y modelos de visión.
pesos = models.ResNet18_Weights.DEFAULT  # Selecciona los pesos preentrenados oficiales.
transformacion = pesos.transforms()  # Aplica el tamaño y normalización esperados por dichos pesos.
entrenamiento = datasets.Flowers102('datos', split='train', download=True, transform=transformacion)  # Descarga y prepara entrenamiento.
validacion = datasets.Flowers102('datos', split='val', download=True, transform=transformacion)  # Carga validación independiente.
prueba = datasets.Flowers102('datos', split='test', download=True, transform=transformacion)  # Carga prueba independiente.
clases = [f'Flor {i+1}' for i in range(102)]  # Usa identificadores de clase; consulten el catálogo para nombres botánicos.
train_loader = DataLoader(entrenamiento, batch_size=32, shuffle=True)  # Forma lotes mezclados para entrenamiento.
val_loader = DataLoader(validacion, batch_size=32)  # Forma lotes para validación.
test_loader = DataLoader(prueba, batch_size=32)  # Forma lotes para prueba.
print('Clases:', clases, '| tamaños:', len(entrenamiento), len(validacion), len(prueba))  # Muestra las clases y particiones.


### Preguntas y actividad obligatoria

1. ¿Qué representa cada conjunto y cuántos ejemplos contiene?
2. Investiguen por qué no debe ajustarse el modelo con prueba. Cambien el tamaño del lote, vuelvan a ejecutar y comparen tiempo y resultados.

**Respuestas del equipo:** escriban aquí sus respuestas y evidencias.

## 3. Arquitectura y entrenamiento

El modelo recibe imágenes preparadas con la transformación asociada a sus pesos.

In [ ]:
modelo = models.resnet18(weights=pesos)  # Carga la red con conocimientos preentrenados.
for parametro in modelo.parameters():  # Recorre todos los parámetros de la red.
    parametro.requires_grad = False  # Impide actualizar los parámetros originales.
entradas = modelo.fc.in_features  # Consulta el tamaño de la representación anterior al clasificador.
modelo.fc = nn.Linear(entradas, len(clases))  # Sustituye la capa final por una adaptada al problema.
modelo = modelo.to(device)  # Coloca la red en el dispositivo disponible.
perdida = nn.CrossEntropyLoss()  # Calcula la pérdida para clasificación multiclase.
optimizador = torch.optim.Adam(modelo.fc.parameters(), lr=0.001)  # Entrena solo la nueva capa final.
epocas = 5  # Define las épocas iniciales.
print('Entrada del clasificador:', entradas, '| clases:', len(clases))  # Muestra la adaptación realizada.


### Preguntas y actividad obligatoria

1. Dibujen la arquitectura e indiquen la forma de entrada y salida de cada componente.
2. Investiguen y justifiquen la pérdida, activación, optimizador y tasa de aprendizaje.
3. Modifiquen un hiperparámetro, reentrenen y comparen con el resultado original.

**Respuestas del equipo:** escriban aquí sus respuestas y evidencias.

## 4. Ciclo de entrenamiento

En entrenamiento se calculan gradientes y se actualizan parámetros. En validación se desactiva el cálculo de gradientes.

In [ ]:
def evaluar(loader):  # Define una función para medir pérdida y exactitud.
    modelo.eval()  # Cambia el modelo al modo de evaluación.
    suma, aciertos, cantidad = 0.0, 0, 0  # Inicializa los acumuladores.
    with torch.no_grad():  # Evita construir el grafo de gradientes.
        for x, y in loader:  # Recorre los lotes del conjunto recibido.
            x, y = x.to(device), y.to(device)  # Mueve entradas y etiquetas al dispositivo.
            logits = modelo(x)  # Calcula las puntuaciones de cada clase.
            suma += perdida(logits, y).item() * y.size(0)  # Acumula la pérdida ponderada por lote.
            aciertos += (logits.argmax(1) == y).sum().item()  # Cuenta predicciones correctas.
            cantidad += y.size(0)  # Cuenta ejemplos procesados.
    return suma / cantidad, aciertos / cantidad  # Devuelve pérdida media y exactitud.
historia = {'train_loss': [], 'val_loss': [], 'val_acc': []}  # Guarda métricas por época.
for epoca in range(epocas):  # Repite el entrenamiento el número de épocas elegido.
    modelo.train()  # Activa el modo de entrenamiento.
    suma, cantidad = 0.0, 0  # Reinicia acumuladores de entrenamiento.
    for x, y in train_loader:  # Recorre los lotes de entrenamiento.
        x, y = x.to(device), y.to(device)  # Lleva el lote al dispositivo.
        optimizador.zero_grad()  # Limpia gradientes de la iteración anterior.
        logits = modelo(x)  # Calcula las puntuaciones de salida.
        error = perdida(logits, y)  # Compara las puntuaciones con las etiquetas.
        error.backward()  # Calcula gradientes mediante backpropagation.
        optimizador.step()  # Actualiza los parámetros entrenables.
        suma += error.item() * y.size(0)  # Acumula la pérdida ponderada.
        cantidad += y.size(0)  # Acumula la cantidad de ejemplos.
    val_loss, val_acc = evaluar(val_loader)  # Evalúa sin actualizar parámetros.
    historia['train_loss'].append(suma / cantidad)  # Registra pérdida de entrenamiento.
    historia['val_loss'].append(val_loss)  # Registra pérdida de validación.
    historia['val_acc'].append(val_acc)  # Registra exactitud de validación.
    print(f'Época {epoca+1}: pérdida={suma/cantidad:.4f}, val={val_loss:.4f}, exactitud={val_acc:.3f}')  # Resume la época.


### Preguntas y actividad obligatoria

1. Expliquen `zero_grad`, `backward` y `step` y qué ocurre si cambia su orden.
2. ¿Qué diferencias hay entre `train`, `eval` y `no_grad`? Investiguen.
3. Dupliquen las épocas y comparen entrenamiento y validación; ¿observan sobreajuste?

**Respuestas del equipo:** escriban aquí sus respuestas y evidencias.

## 5. Curvas y prueba

Utilicen la prueba solo después de tomar sus decisiones con validación.

In [ ]:
plt.plot(historia['train_loss'], label='Entrenamiento')  # Dibuja la pérdida de entrenamiento.
plt.plot(historia['val_loss'], label='Validación')  # Dibuja la pérdida de validación.
plt.xlabel('Época')  # Nombra el eje horizontal.
plt.ylabel('Pérdida')  # Nombra el eje vertical.
plt.legend()  # Identifica las curvas.
plt.show()  # Muestra la gráfica.
test_loss, test_acc = evaluar(test_loader)  # Evalúa una vez el conjunto reservado.
print('Prueba: pérdida', round(test_loss, 4), 'exactitud', round(test_acc, 4))  # Muestra los resultados.


### Preguntas y actividad obligatoria

1. ¿Qué tendencia tienen las curvas? ¿Hay evidencia de sobreajuste?
2. ¿Qué significa la exactitud y qué información no proporciona? Investíguenlo.
3. Registren los resultados de sus experimentos y justifiquen su configuración final.

**Respuestas del equipo:** escriban aquí sus respuestas y evidencias.

## 6. Errores y matriz de confusión

Las filas indican la clase real y las columnas la predicción.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report  # Importa herramientas de evaluación.
reales, predichos = [], []  # Prepara listas para etiquetas y predicciones.
modelo.eval()  # Activa el modo de evaluación.
with torch.no_grad():  # Desactiva el cálculo de gradientes.
    for x, y in test_loader:  # Recorre el conjunto de prueba.
        pred = modelo(x.to(device)).argmax(1).cpu()  # Obtiene la clase más probable.
        reales.extend(y.tolist())  # Guarda las etiquetas verdaderas.
        predichos.extend(pred.tolist())  # Guarda las clases predichas.
matriz = confusion_matrix(reales, predichos, labels=list(range(len(clases))))  # Cuenta aciertos y confusiones.
ConfusionMatrixDisplay(matriz, display_labels=clases).plot(xticks_rotation=90, cmap='Blues')  # Grafica la matriz.
plt.show()  # Muestra la matriz.
print(classification_report(reales, predichos, labels=list(range(len(clases))), target_names=clases, zero_division=0))  # Muestra métricas por clase.


### Preguntas y actividad obligatoria

1. Identifiquen dos confusiones frecuentes y propongan una explicación.
2. Investiguen precisión, recall y F1; ¿qué aporta cada una?
3. Seleccionen al menos tres errores y expliquen por qué podrían ser difíciles.

**Respuestas del equipo:** escriban aquí sus respuestas y evidencias.

## 7. Guardado, carga e inferencia

El archivo contiene pesos y metadatos para reproducir la configuración. Conserven también el código y el preprocesamiento.

In [ ]:
ruta_modelo = Path('modelo_entrenado.pth')  # Define el nombre del archivo de salida.
torch.save({'pesos': modelo.state_dict(), 'clases': clases}, ruta_modelo)  # Guarda pesos y nombres de clases.
checkpoint = torch.load(ruta_modelo, map_location=device, weights_only=True)  # Carga el archivo confiable en el dispositivo elegido.
modelo.load_state_dict(checkpoint['pesos'])  # Restaura los parámetros en la arquitectura definida.
modelo.eval()  # Prepara la red para inferencia.
x, y = next(iter(test_loader))  # Obtiene un lote que no se usó para entrenar.
with torch.no_grad():  # Desactiva gradientes durante la predicción.
    probabilidades = torch.softmax(modelo(x[:1].to(device)), dim=1)[0].cpu()  # Convierte puntuaciones en probabilidades normalizadas.
indice = probabilidades.argmax().item()  # Localiza la clase con mayor puntuación.
print('Real:', clases[y[0].item()], '| Predicción:', clases[indice], '| Confianza:', round(probabilidades[indice].item(), 3))  # Presenta el resultado.


### Preguntas y actividad obligatoria

1. ¿Por qué deben conservar arquitectura y preprocesamiento además de pesos?
2. ¿Qué representa softmax y por qué una confianza alta puede ser incorrecta?
3. Prueben al menos cinco ejemplos, registren resultados y expliquen errores. Investiguen cómo descongelar layer4 y reconstruir el optimizador; comparen con la capa congelada.

**Respuestas del equipo:** escriban aquí sus respuestas y evidencias.

## Entrega y defensa

Entreguen el notebook ejecutado, modelo, fuentes y reporte. Incluyan una tabla con configuración inicial, modificaciones, métricas y explicación. Respondan todas las preguntas, citen la documentación consultada y expliquen qué cambiarían. No seleccionen hiperparámetros usando prueba.

## 8. Tres clases más probables

Top-3 muestra alternativas; no garantiza que las probabilidades estén calibradas.

In [ ]:
x, y = next(iter(test_loader))  # Obtiene un lote de prueba.
with torch.no_grad():  # Evita gradientes durante inferencia.
    p = torch.softmax(modelo(x[:1].to(device)), dim=1)[0].cpu()  # Calcula puntuaciones normalizadas.
valores, indices = p.topk(3)  # Selecciona las tres puntuaciones mayores.
for valor, indice in zip(valores, indices):  # Recorre puntuaciones y clases seleccionadas.
    print(clases[indice.item()], round(valor.item(), 3))  # Muestra cada clase y su confianza.
print('Clase real:', clases[y[0].item()])  # Muestra la etiqueta verdadera.


### Preguntas y actividad obligatoria

1. ¿La clase real aparece entre las tres primeras?
2. Investiguen Top-1 y Top-3 y calculen Top-3 para toda la prueba. Justifiquen su interpretación.

**Respuestas del equipo:** escriban aquí sus respuestas y evidencias.